# Lab 3: Context Engineering — Retrieval, Memory, and State

A language model's parameters are not a reliable database. Agentic systems therefore need deliberate mechanisms for selecting external evidence, managing a limited context window, preserving task state, and deciding what should persist across interactions.

In this lab, we will build a small retrieval-augmented generation (RAG) workflow over a synthetic course-policy collection. Embeddings and retrieval run locally using the free `sentence-transformers/all-MiniLM-L6-v2` model. `gpt-5-nano` is used only to answer from retrieved evidence.

No Canvas content, student records, or private data are used. **Note:** This session is graded, so complete all exercises within class and uploade the saved notebook with outputs to Canvas. 

## References and further reading

1. Roitman, H. (2026). [*The Hitchhiker's Guide to Agentic AI: From Foundations to Systems*](https://arxiv.org/abs/2606.24937), Chapters 16–17 on retrieval-augmented generation and agentic memory, plus Sections 18.2 and 18.6 on context-window and state management.
2. Anthropic. [Effective context engineering for AI agents](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents). Focus on context as a finite resource, context selection, compaction, memory, tool results, and the distinction between context engineering and prompt engineering.
3. Anthropic. [Contextual Retrieval](https://www.anthropic.com/engineering/contextual-retrieval). Read as a practical discussion of lost chunk context, lexical and semantic retrieval, hybrid approaches, reranking, and retrieval evaluation.
4. Sentence Transformers. [Retrieve and re-rank](https://www.sbert.net/examples/sentence_transformer/applications/retrieve_rerank/README.html). Connects bi-encoder semantic retrieval with cross-encoder reranking and explains why top retrieval scores do not guarantee relevance.
5. OpenAI. [`text-embedding-3-small` model documentation](https://developers.openai.com/api/docs/models/text-embedding-3-small). Use for the optional comparison with the small OpenAI embedding model; check current endpoint support and cost before running calls.
6. OpenAI. [GPT-5 nano model documentation](https://developers.openai.com/api/docs/models/gpt-5-nano). Consult for the grounded-generation model's current Responses API support, limits, and rate information.
7. LangGraph. [Persistence](https://docs.langchain.com/oss/python/langgraph/persistence). Focus on checkpoints, threads, fault tolerance, replay, and the operational consequences of persisting workflow state.
8. LangGraph. [Memory](https://docs.langchain.com/oss/python/langgraph/add-memory). Distinguishes short-term thread memory from long-term cross-session memory and demonstrates storage, update, and deletion patterns.
9. Nogueira and Cho (2019). [Passage Re-ranking with BERT](https://arxiv.org/abs/1901.04085). A concise foundational paper on using a cross-encoder-style model to rerank candidates from a first-stage retriever.


## 0. Install and import the required libraries

Sentence Transformers supplies a free local embedding model. NumPy performs exact cosine-similarity search, which is sufficient for this small teaching collection. Larger systems may use FAISS, Chroma, Qdrant, pgvector, or another vector store.

In [ ]:
%pip install -q --upgrade openai sentence-transformers pandas numpy

In [ ]:
import os
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from openai import OpenAI

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GENERATION_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-nano")
print("Embedding model:", EMBEDDING_MODEL)
print("Generation model:", GENERATION_MODEL)

## 1. Context engineering, retrieval, memory, and state

These terms are related but not interchangeable:

- **Context engineering** is the design of what information and instructions enter a model call, in what form, and within what budget.
- **Retrieval** selects potentially relevant information from an external collection.
- **Working memory** is information needed during the current interaction or task.
- **Persistent memory** is information deliberately stored for later interactions.
- **State** records where a workflow is and what has already happened—for example, the current question, retrieved chunk IDs, clarification status, and call count.

A good system does not dump everything into the prompt or remember everything forever. It selects, budgets, cites, updates, and forgets deliberately.

## 2. Retrieval 1.0: Create a document collection with provenance

Every document receives a stable ID and source label. Provenance lets us trace a generated claim back to evidence. The policies below are synthetic and exist only for this lab.

In [ ]:
DOCUMENTS = [
    {
        "doc_id": "POLICY-AI",
        "title": "Generative AI Use",
        "source": "Synthetic course policy",
        "text": (
            "Tier 0 activities prohibit generative AI, AI-enabled search, and AI code completion. "
            "The paper-based quiz is Tier 0. Tier 1 activities permit only the models, tools, "
            "and stages explicitly authorized in the instructions. Students must preserve requested "
            "prompts, outputs, traces, and disclosures. Tier 2 permits broader AI use for projects, "
            "subject to data, security, documentation, and resource limits. Students remain responsible "
            "for every submitted result."
        ),
    },
    {
        "doc_id": "POLICY-ACCESS",
        "title": "Model Access and Credits",
        "source": "Synthetic course policy",
        "text": (
            "Required agentic interactions use an instructor-managed mini-model service with capped "
            "course credit. Credentials may not be shared and usage limits may not be bypassed. The course "
            "and School do not sponsor optional commercial subscriptions, premium plans, or additional API "
            "credits. Purchasing optional access is not necessary for course requirements."
        ),
    },
    {
        "doc_id": "POLICY-LATE",
        "title": "Late Work and Grace Period",
        "source": "Synthetic course policy",
        "text": (
            "A total grace allowance of two days applies to eligible assignments. A student may use one "
            "two-day extension or two separate one-day extensions. Students should contact the instructor "
            "as soon as possible and preferably before the deadline. The grace allowance does not "
            "automatically apply to presentations, in-class activities, or team milestones that affect others."
        ),
    },
    {
        "doc_id": "POLICY-PROJECTS",
        "title": "Project Deliverables",
        "source": "Synthetic course policy",
        "text": (
            "Project 1 requires a working tool-using workflow, evaluation artifacts, a presentation and "
            "demonstration, and a technical report. Project 2 requires a working production-oriented "
            "agentic system, evaluation evidence, an architecture defense, a security and human-oversight "
            "analysis, a presentation and demonstration, and a final technical report."
        ),
    },
    {
        "doc_id": "POLICY-DATA",
        "title": "Data and Course Materials",
        "source": "Synthetic course policy",
        "text": (
            "Do not submit confidential, proprietary, personally identifiable, export-controlled, or "
            "otherwise restricted data to an AI system. Course materials may not be uploaded to an external "
            "AI tool unless the instructor explicitly authorizes the material and tool for that activity. "
            "Authorization to use an AI tool does not authorize uploading all course content."
        ),
    },
]

pd.DataFrame(DOCUMENTS)[["doc_id", "title", "source"]]

## 3. Retrieval 2.0: Compare two chunking strategies

A retriever searches chunks, not abstract documents. Chunk boundaries determine what can be found and what context remains together. We will compare:

1. **Whole-document chunks:** simple and provenance-friendly, but potentially broad.
2. **Sentence-window chunks:** more precise, but surrounding context may be fragmented.

*[Question: What happens if an answer requires two sentences that are split across chunks?]*

In [ ]:
def split_sentences(text: str) -> list[str]:
    return [sentence.strip() for sentence in re.split(r"(?<=[.!?])\s+", text) if sentence.strip()]

def whole_document_chunks(documents: list[dict]) -> list[dict]:
    return [
        {
            "chunk_id": f"{doc['doc_id']}-C01",
            "doc_id": doc["doc_id"],
            "title": doc["title"],
            "source": doc["source"],
            "text": doc["text"],
        }
        for doc in documents
    ]

def sentence_window_chunks(documents: list[dict], window_size: int = 2) -> list[dict]:
    chunks = []
    for doc in documents:
        sentences = split_sentences(doc["text"])
        for start in range(0, len(sentences), window_size):
            window = sentences[start:start + window_size]
            chunks.append({
                "chunk_id": f"{doc['doc_id']}-C{len(chunks) + 1:02d}",
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "source": doc["source"],
                "text": " ".join(window),
            })
    return chunks

whole_chunks = whole_document_chunks(DOCUMENTS)
window_chunks = sentence_window_chunks(DOCUMENTS, window_size=2)
print("Whole-document chunks:", len(whole_chunks))
print("Sentence-window chunks:", len(window_chunks))
pd.DataFrame(window_chunks).head()

## 4. Retrieval 3.0: Embed chunks locally

An embedding maps text to a numeric vector. Texts with similar meanings tend to have nearby vectors. `all-MiniLM-L6-v2` is small, free to run locally, and suitable for a teaching-scale semantic-search demonstration.

We normalize vectors so their dot product is cosine similarity. The first execution downloads the model files.

In [ ]:
embedder = SentenceTransformer(EMBEDDING_MODEL)

def build_index(chunks: list[dict]) -> dict:
    texts = [chunk["text"] for chunk in chunks]
    embeddings = embedder.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return {"chunks": chunks, "embeddings": np.asarray(embeddings)}

whole_index = build_index(whole_chunks)
window_index = build_index(window_chunks)
print("Embedding shape for whole-document index:", whole_index["embeddings"].shape)
print("Embedding shape for sentence-window index:", window_index["embeddings"].shape)

## 5. Retrieval 4.0: Perform dense semantic search

The retriever embeds the question, calculates similarity against every chunk, and returns the top results with scores and provenance. High similarity is evidence of semantic closeness—not proof that the chunk answers the question.

In [ ]:
def retrieve(query: str, index: dict, top_k: int = 3) -> list[dict]:
    query_vector = embedder.encode([query], normalize_embeddings=True, show_progress_bar=False)[0]
    scores = index["embeddings"] @ query_vector
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [
        {**index["chunks"][i], "score": float(scores[i])}
        for i in top_indices
    ]

query = "Can I use AI during the paper quiz?"
results = retrieve(query, window_index, top_k=3)
pd.DataFrame(results)[["chunk_id", "title", "score", "text"]]

## 6. Retrieval 5.0: Compare chunking strategies

We will run the same query against both indexes. Look beyond the first rank: does either strategy return redundant, incomplete, or overly broad evidence?

In [ ]:
comparison_rows = []
for strategy, index in [("whole_document", whole_index), ("sentence_window", window_index)]:
    for rank, result in enumerate(retrieve(query, index, top_k=3), start=1):
        comparison_rows.append({
            "strategy": strategy,
            "rank": rank,
            "chunk_id": result["chunk_id"],
            "score": round(result["score"], 3),
            "text": result["text"],
        })

pd.DataFrame(comparison_rows)

## 6A. Retrieval 6.0: Why dense retrieval is not enough

Dense retrieval is strong when a question and a passage express the same idea using different words. It can still miss exact identifiers, unusual names, error codes, and rare phrases. **Sparse retrieval** rewards exact token overlap instead.

The two methods fail differently:

- Dense retrieval may connect *subscription funding* with *premium plan payment* even when the words differ.
- Sparse retrieval is often better for literal strings such as `Tier 0`, `POLICY-DATA`, or a product code.
- Hybrid retrieval combines both signals instead of assuming one retriever is universally best.

Below, we implement a transparent BM25-style sparse retriever. BM25 scores a chunk using query-term frequency, term rarity across the collection, and document-length normalization. This implementation is intentionally small enough to inspect line by line.

In [ ]:
from collections import Counter
import math

def tokenize_for_search(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())

def build_sparse_index(chunks: list[dict]) -> dict:
    tokenized = [tokenize_for_search(chunk["text"]) for chunk in chunks]
    document_frequency = Counter()
    for tokens in tokenized:
        document_frequency.update(set(tokens))
    return {
        "chunks": chunks,
        "tokenized": tokenized,
        "document_frequency": document_frequency,
        "average_length": np.mean([len(tokens) for tokens in tokenized]),
    }

def sparse_retrieve(query: str, index: dict, top_k: int = 3, k1: float = 1.5, b: float = 0.75) -> list[dict]:
    query_terms = tokenize_for_search(query)
    number_of_chunks = len(index["chunks"])
    scores = []
    for tokens in index["tokenized"]:
        frequencies = Counter(tokens)
        score = 0.0
        for term in query_terms:
            term_frequency = frequencies[term]
            if term_frequency == 0:
                continue
            doc_frequency = index["document_frequency"][term]
            inverse_doc_frequency = math.log(1 + (number_of_chunks - doc_frequency + 0.5) / (doc_frequency + 0.5))
            length_normalizer = term_frequency + k1 * (1 - b + b * len(tokens) / index["average_length"])
            score += inverse_doc_frequency * (term_frequency * (k1 + 1)) / length_normalizer
        scores.append(score)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [{**index["chunks"][i], "score": float(scores[i])} for i in top_indices]

sparse_index = build_sparse_index(window_chunks)
exact_term_query = "What activities are Tier 0?"
pd.DataFrame(sparse_retrieve(exact_term_query, sparse_index))[
    ["chunk_id", "title", "score", "text"]
]

## 6B. Retrieval 7.0: Combine rankings with reciprocal rank fusion

Dense and sparse scores are not directly comparable: cosine similarity and BM25 live on different scales. **Reciprocal Rank Fusion (RRF)** avoids fragile score normalization by combining ranks. A chunk receives `1 / (k + rank)` from each list. Chunks supported by both retrievers rise in the combined ranking.

RRF is not magic. It adds latency and can amplify a shared error. It is useful because it is simple, explainable, and robust across differently scaled retrievers.

In [ ]:
def reciprocal_rank_fusion(result_lists: list[list[dict]], top_k: int = 3, rrf_k: int = 60) -> list[dict]:
    fused_scores = Counter()
    chunks_by_id = {}
    component_ranks = {}
    for list_number, results_list in enumerate(result_lists):
        for rank, result in enumerate(results_list, start=1):
            chunk_id = result["chunk_id"]
            chunks_by_id[chunk_id] = result
            fused_scores[chunk_id] += 1 / (rrf_k + rank)
            component_ranks.setdefault(chunk_id, {})[f"list_{list_number}"] = rank
    ranked_ids = sorted(fused_scores, key=fused_scores.get, reverse=True)[:top_k]
    return [
        {**chunks_by_id[chunk_id], "score": fused_scores[chunk_id], "component_ranks": component_ranks[chunk_id]}
        for chunk_id in ranked_ids
    ]

def hybrid_retrieve(query: str, dense_index: dict, sparse_index: dict, top_k: int = 3) -> list[dict]:
    candidate_count = min(max(top_k * 3, 10), len(dense_index["chunks"]))
    dense_results = retrieve(query, dense_index, top_k=candidate_count)
    sparse_results = sparse_retrieve(query, sparse_index, top_k=candidate_count)
    return reciprocal_rank_fusion([dense_results, sparse_results], top_k=top_k)

for method_name, method_results in {
    "dense": retrieve(exact_term_query, window_index),
    "sparse": sparse_retrieve(exact_term_query, sparse_index),
    "hybrid": hybrid_retrieve(exact_term_query, window_index, sparse_index),
}.items():
    print(method_name, [item["chunk_id"] for item in method_results])

## 6C. Retrieval 8.0: Filter with metadata before ranking

An embedding does not understand authorization boundaries. If a user may search only a subset of records, access control must happen **before** those records can become candidates or enter model context. Filtering after generation is too late.

Our synthetic corpus has no restricted records, so the example filters by document ID to demonstrate the mechanism. In production, metadata might represent tenant, owner, confidentiality level, date, document type, or retention status.

In [ ]:
def filter_index(index: dict, allowed_doc_ids: set[str]) -> dict:
    selected_positions = [
        position for position, chunk in enumerate(index["chunks"])
        if chunk["doc_id"] in allowed_doc_ids
    ]
    return {
        "chunks": [index["chunks"][position] for position in selected_positions],
        "embeddings": index["embeddings"][selected_positions],
    }

access_only_index = filter_index(window_index, {"POLICY-ACCESS"})
filtered_results = retrieve("What is the late-work policy?", access_only_index, top_k=3)
pd.DataFrame(filtered_results)[["chunk_id", "title", "score"]]

## 7. Context engineering 1.0: Assemble evidence within a budget

Retrieval returns candidates. Context assembly decides what actually enters the model call. We will enforce a simple character budget, avoid duplicate chunks, and retain visible chunk identifiers for citation.

Character counts are only a teaching approximation for token budgets. Production systems should use the model's tokenizer or API token accounting.

In [ ]:
def assemble_context(results: list[dict], max_characters: int = 1400) -> dict:
    blocks = []
    included_ids = []
    used = 0
    seen_text = set()

    for result in results:
        if result["text"] in seen_text:
            continue
        block = f"[{result['chunk_id']}] {result['title']}\n{result['text']}"
        if used + len(block) > max_characters:
            continue
        blocks.append(block)
        included_ids.append(result["chunk_id"])
        seen_text.add(result["text"])
        used += len(block)

    return {
        "context": "\n\n".join(blocks),
        "included_chunk_ids": included_ids,
        "characters_used": used,
        "character_budget": max_characters,
    }

context_package = assemble_context(results, max_characters=900)
print(json.dumps({k: v for k, v in context_package.items() if k != "context"}, indent=2))
print("\nCONTEXT SENT TO MODEL:\n")
print(context_package["context"] )

## 7A. Context engineering 2.0: Selection, ordering, and compression

A context window is a budget, not a target. The next section demonstrates why selection and compression must be deliberate.

### Selection, ordering, and compression

A context window is a budget, not a target. Filling it with every available passage can increase cost, bury decisive evidence, repeat claims, and introduce contradictions. A context assembler therefore makes several policy choices:

1. **Selection:** Which candidates are relevant and authorized?
2. **Ordering:** Should the most useful evidence appear first, last, or near the question?
3. **Deduplication:** Are multiple chunks saying the same thing?
4. **Compression:** Can irrelevant sentences be removed without changing evidentiary meaning?
5. **Provenance:** Can every retained statement still be traced to its source?

Compression is risky when performed by a generative model because the summary can distort exceptions or invent connections. The extractive compressor below keeps original sentences and therefore preserves a direct evidence trail.

In [ ]:
def extractively_compress(query: str, results: list[dict], sentences_per_chunk: int = 1) -> list[dict]:
    query_terms = set(tokenize_for_search(query))
    compressed_results = []
    for result in results:
        sentences = split_sentences(result["text"])
        scored_sentences = []
        for original_position, sentence in enumerate(sentences):
            sentence_terms = set(tokenize_for_search(sentence))
            overlap = len(query_terms & sentence_terms)
            scored_sentences.append((overlap, -original_position, sentence))
        selected = sorted(scored_sentences, reverse=True)[:sentences_per_chunk]
        selected_in_original_order = [item[2] for item in sorted(selected, key=lambda item: -item[1])]
        compressed_results.append({
            **result,
            "original_text": result["text"],
            "text": " ".join(selected_in_original_order),
            "compression_method": "extractive_query_overlap",
        })
    return compressed_results

compression_query = "Can the grace days apply to a presentation?"
uncompressed = retrieve(compression_query, whole_index, top_k=2)
compressed = extractively_compress(compression_query, uncompressed)
for before, after in zip(uncompressed, compressed):
    print(before["chunk_id"])
    print("BEFORE:", before["text"])
    print("AFTER: ", after["text"], "\n")

### A warning about lexical compression

The compressor is deliberately imperfect. A sentence containing a critical exception may share few words with the question and be dropped. For example, the phrase *does not automatically apply* changes the meaning of the grace-period policy. Inspect whether the compressor preserves it.

*[Question: Which is worse for this application—irrelevant extra context or removing a low-overlap exception? How would you test that tradeoff?]*

## 8. Grounded generation 1.0: Answer only from retrieved evidence

The model receives explicit evidence and instructions to cite chunk IDs and abstain when the evidence is insufficient. This reduces unsupported answers, but the instruction alone does not guarantee grounding. We must evaluate the answer and citations.

In [ ]:
if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError("OPENAI_API_KEY is not set. Use the instructor-managed access instructions.")

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL") or None,
)

def generate_grounded_answer(question: str, context: str) -> str:
    response = client.responses.create(
        model=GENERATION_MODEL,
        instructions=(
            "Answer only from the supplied evidence. Cite supporting chunk IDs in square "
            "brackets. If the evidence does not answer the question, say exactly: "
            "'Insufficient evidence in the provided collection.' Do not use outside knowledge."
        ),
        input=f"QUESTION:\n{question}\n\nEVIDENCE:\n{context}",
    )
    return response.output_text

answer = generate_grounded_answer(query, context_package["context"])
print(answer)

## 9. Grounded generation 2.0: Build a complete RAG trace

A useful trace records the question, retrieved candidates, selected context, answer, and citations. This lets us distinguish retrieval failure from generation failure.

In [ ]:
def answer_with_rag(question: str, index: dict, top_k: int = 3, max_characters: int = 1400) -> dict:
    retrieved = retrieve(question, index=index, top_k=top_k)
    package = assemble_context(retrieved, max_characters=max_characters)
    answer_text = generate_grounded_answer(question, package["context"])
    cited_ids = re.findall(r"\[([A-Z0-9-]+-C[0-9]+)\]", answer_text)
    return {
        "question": question,
        "retrieved": retrieved,
        "included_chunk_ids": package["included_chunk_ids"],
        "characters_used": package["characters_used"],
        "answer": answer_text,
        "cited_chunk_ids": cited_ids,
    }

rag_trace = answer_with_rag("Does the School pay for my optional premium AI subscription?", window_index)
print(rag_trace["answer"])
print("Included:", rag_trace["included_chunk_ids"])
print("Cited:", rag_trace["cited_chunk_ids"] )

## 10. Grounded generation 3.0: Test abstention

The collection contains no parking policy. A safe workflow should not answer from general knowledge. Retrieval will still return the closest available chunks, which is why generation must be instructed and evaluated for abstention.

In [ ]:
unsupported_trace = answer_with_rag("Where may students park after 5 PM?", window_index)
print(unsupported_trace["answer"])
print("\nTop retrieved chunks:")
for result in unsupported_trace["retrieved"]:
    print(result["chunk_id"], round(result["score"], 3), result["title"])

## 11. State 1.0: Represent workflow state explicitly

State is not the same as conversational prose. An explicit state object makes the workflow inspectable and testable. Here we record the current question, retrieval configuration, evidence IDs, answer, status, and call count.

In [ ]:
def create_task_state(question: str) -> dict:
    return {
        "question": question,
        "status": "created",
        "top_k": 3,
        "context_budget": 1400,
        "retrieved_chunk_ids": [],
        "answer": None,
        "model_calls": 0,
        "errors": [],
    }

def run_stateful_rag(state: dict, index: dict) -> dict:
    state = dict(state)
    state["status"] = "retrieving"
    retrieved = retrieve(state["question"], index, top_k=state["top_k"])
    package = assemble_context(retrieved, state["context_budget"])
    state["retrieved_chunk_ids"] = package["included_chunk_ids"]
    state["status"] = "generating"
    state["answer"] = generate_grounded_answer(state["question"], package["context"])
    state["model_calls"] += 1
    state["status"] = "completed"
    return state

task_state = create_task_state("How may the two grace days be used?")
completed_state = run_stateful_rag(task_state, window_index)
print(json.dumps(completed_state, indent=2))

## 11A. State 2.0: Checkpoints and resumability

Explicit state also makes interruption and recovery possible. A workflow can checkpoint a safe subset of its state and resume from a known stage instead of starting over.

The checkpoint below uses synthetic data and a versioned envelope. Production checkpoints additionally require task isolation, access controls, retention limits, and secret filtering.

### A minimal checkpoint implementation

A long-running agent may fail after retrieval but before generation, or pause while waiting for human approval. A checkpoint records enough state to resume without repeating completed work. It should also include a schema version so future code can recognize or migrate older records.

Checkpointing creates obligations: secrets should be excluded, writes should be atomic, users and tasks should be isolated, and stale state should expire. The local example below contains only synthetic data.

In [ ]:
CHECKPOINT_PATH = Path("Week3_demo_checkpoint.json")

def save_checkpoint(state: dict, path: Path = CHECKPOINT_PATH) -> None:
    safe_state = {key: value for key, value in state.items() if key not in {"api_key", "raw_private_data"}}
    payload = {"schema_version": 1, "state": safe_state}
    temporary_path = path.with_suffix(".tmp")
    temporary_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    temporary_path.replace(path)

def load_checkpoint(path: Path = CHECKPOINT_PATH) -> dict:
    payload = json.loads(path.read_text(encoding="utf-8"))
    if payload.get("schema_version") != 1:
        raise ValueError("Unsupported checkpoint schema")
    return payload["state"]

checkpoint_demo = create_task_state("What evidence is required for Project 2?")
checkpoint_demo["status"] = "awaiting_retrieval"
save_checkpoint(checkpoint_demo)
print(load_checkpoint())

## 12. Memory 1.0: Separate working memory from persistent memory

The completed task state is working memory for one interaction. We should not automatically persist the entire prompt, retrieved evidence, or answer. Persistent memory should be minimal, purposeful, reviewable, and removable.

We will store only an explicit, non-sensitive user preference in a local JSON file. In a real system, saving a preference should require notice, consent, access control, and a deletion path.

In [ ]:
MEMORY_PATH = Path("Week3_demo_memory.json")

def load_memory(path: Path = MEMORY_PATH) -> dict:
    if not path.exists():
        return {"preferences": {}}
    return json.loads(path.read_text(encoding="utf-8"))

def save_preference(key: str, value: str, path: Path = MEMORY_PATH) -> dict:
    allowed_keys = {"answer_style", "citation_style"}
    if key not in allowed_keys:
        raise ValueError(f"Preference not allowed: {key}")
    memory = load_memory(path)
    memory["preferences"][key] = value
    path.write_text(json.dumps(memory, indent=2), encoding="utf-8")
    return memory

def clear_memory(path: Path = MEMORY_PATH) -> None:
    if path.exists():
        path.unlink()

memory = save_preference("answer_style", "concise")
print(memory)
print("Memory can be removed with clear_memory().")

## 12A. Memory 2.0: Working, episodic, semantic, and procedural memory

Agent literature borrows several memory labels from cognitive science. They are useful design metaphors, not proof that a system remembers like a person.

| Memory type | Agentic interpretation | Example | Typical lifetime |
|---|---|---|---|
| Working memory | Current task state placed in or near the active context | question, retrieved IDs, pending approval | one task or session |
| Episodic memory | Records of particular prior interactions or outcomes | a tool call failed because an argument was malformed | bounded history |
| Semantic memory | Extracted facts or stable preferences | user prefers concise answers | until updated or deleted |
| Procedural memory | Instructions governing how the system operates | tool policy, workflow graph, system instructions | versioned with the system |

Persistent storage is not automatically memory, and more memory is not automatically better. A record should have a purpose, owner, provenance, retention period, access rule, update rule, and deletion path. Sensitive attributes should not be inferred and stored merely because they might personalize an answer.

In [ ]:
MEMORY_CANDIDATES = [
    {"item": "current question", "type": "working", "decision": "temporary"},
    {"item": "retrieved chunk IDs", "type": "working", "decision": "temporary"},
    {"item": "explicit concise-answer preference", "type": "semantic", "decision": "persist with consent"},
    {"item": "tool failure and remediation", "type": "episodic", "decision": "retain briefly"},
    {"item": "system tool-use policy", "type": "procedural", "decision": "version with application"},
    {"item": "inferred medical condition", "type": "sensitive", "decision": "reject"},
]
pd.DataFrame(MEMORY_CANDIDATES)

## 12B. Memory 3.0: Retrieval, conflict, and forgetting

A memory system needs more than `save()`. It must decide when to retrieve a record, what to do when records conflict, and when to forget. Useful policies include:

- prefer an explicit newer preference over an older preference;
- never let a remembered preference override safety or authorization rules;
- show users what is stored and let them correct or delete it;
- attach timestamps, provenance, and confidence to extracted memories;
- expire episodic records that no longer serve their stated purpose; and
- keep one user's memory out of another user's retrieval candidates.

*[Question: If a user once asked for short answers but now requests a detailed tutorial, which instruction should win, and why?]*

## 13. Evaluation 1.0: Build a small RAG test set

We will define expected source documents and whether the system should abstain. This lets us evaluate retrieval separately from final-answer behavior.

In [ ]:
EVALUATION_SET = [
    {"question": "Can AI be used on the paper quiz?", "expected_doc_id": "POLICY-AI", "should_abstain": False},
    {"question": "Who pays for an optional premium AI plan?", "expected_doc_id": "POLICY-ACCESS", "should_abstain": False},
    {"question": "Can I split the grace period across assignments?", "expected_doc_id": "POLICY-LATE", "should_abstain": False},
    {"question": "Does Project 2 require a security analysis?", "expected_doc_id": "POLICY-PROJECTS", "should_abstain": False},
    {"question": "May I upload personal records to an AI service?", "expected_doc_id": "POLICY-DATA", "should_abstain": False},
    {"question": "Where is the final examination held?", "expected_doc_id": None, "should_abstain": True},
]

def retrieval_hit(question: str, expected_doc_id: str | None, index: dict, top_k: int = 3) -> tuple[bool, list[dict]]:
    results = retrieve(question, index, top_k=top_k)
    if expected_doc_id is None:
        return True, results
    return any(result["doc_id"] == expected_doc_id for result in results), results

retrieval_rows = []
for case in EVALUATION_SET:
    hit, retrieved = retrieval_hit(case["question"], case["expected_doc_id"], window_index)
    retrieval_rows.append({
        "question": case["question"],
        "expected_doc_id": case["expected_doc_id"],
        "top_doc_ids": [item["doc_id"] for item in retrieved],
        "retrieval_hit_at_3": hit,
    })

retrieval_df = pd.DataFrame(retrieval_rows)
display(retrieval_df)
print("Retrieval hit@3:", retrieval_df["retrieval_hit_at_3"].mean())

## 14. Evaluation 2.0: Evaluate citations and abstention

This cell incurs one nano-model call per evaluation question. Run it only when ready. Citation validity checks whether cited IDs were actually included in context. Abstention is checked with a simple deterministic phrase match. These metrics are useful but incomplete.

In [ ]:
generation_rows = []
for case in EVALUATION_SET:
    trace = answer_with_rag(case["question"], window_index)
    abstained = "insufficient evidence" in trace["answer"].lower()
    citations_valid = set(trace["cited_chunk_ids"]).issubset(set(trace["included_chunk_ids"]))
    generation_rows.append({
        "question": case["question"],
        "should_abstain": case["should_abstain"],
        "abstained": abstained,
        "abstention_correct": abstained == case["should_abstain"],
        "citations_valid": citations_valid,
        "answer": trace["answer"],
    })

generation_df = pd.DataFrame(generation_rows)
display(generation_df)
print("Abstention accuracy:", generation_df["abstention_correct"].mean())
print("Citation-validity rate:", generation_df["citations_valid"].mean())

## 15. Evaluation 3.0: Diagnose the stage that failed

A final answer marked simply *wrong* provides little engineering guidance. RAG failures should be localized:

1. **Corpus failure:** the needed evidence was never collected or is stale.
2. **Chunking failure:** the answer and its qualification were separated.
3. **Retrieval failure:** an adequate chunk exists but was not ranked high enough.
4. **Selection failure:** the chunk was retrieved but excluded by a budget, filter, or deduplicator.
5. **Compression failure:** decisive language was removed or distorted.
6. **Generation failure:** sufficient context was supplied, but the answer ignored or misrepresented it.
7. **Citation failure:** the answer is plausible but cites no evidence, the wrong evidence, or evidence it never received.
8. **Memory/state failure:** stale, cross-user, or incorrectly resumed state influenced the answer.

The trace created earlier exposes the boundary between retrieval, context assembly, and generation. Production traces should avoid logging secrets and private content.

In [ ]:
def diagnose_trace(trace: dict, expected_doc_id: str | None) -> dict:
    retrieved_doc_ids = [item["doc_id"] for item in trace["retrieved"]]
    included_ids = set(trace["included_chunk_ids"])
    cited_ids = set(trace["cited_chunk_ids"])
    return {
        "expected_source_retrieved": expected_doc_id is None or expected_doc_id in retrieved_doc_ids,
        "all_citations_were_in_context": cited_ids.issubset(included_ids),
        "answer_has_a_citation": bool(cited_ids),
        "abstained": "insufficient evidence" in trace["answer"].lower(),
        "retrieved_doc_ids": retrieved_doc_ids,
    }

# Reuse a trace already generated above; this cell makes no additional API call.
diagnose_trace(rag_trace, expected_doc_id="POLICY-ACCESS")

## 16. Evaluation 4.0: Calibrate an abstention threshold

The instruction to abstain is only one control. A system may also refuse generation when retrieval confidence is too low. However, cosine similarity is not a probability and has no universal cutoff. A threshold must be calibrated on representative supported and unsupported questions, then monitored as the corpus and embedder change.

The following cell displays top dense scores. Do **not** choose a threshold by looking only at one convenient example. Look for overlap between supported and unsupported score distributions, and remember that a tiny teaching set usually makes separation appear easier than it is.

In [ ]:
THRESHOLD_CASES = [
    ("Can AI be used on the paper quiz?", True),
    ("Who pays for optional premium subscriptions?", True),
    ("May grace days be split?", True),
    ("Where can students park?", False),
    ("What food is served in the cafeteria?", False),
    ("When does the campus bus arrive?", False),
]

threshold_rows = []
for threshold_question, is_supported in THRESHOLD_CASES:
    top_result = retrieve(threshold_question, window_index, top_k=1)[0]
    threshold_rows.append({
        "question": threshold_question,
        "supported": is_supported,
        "top_score": round(top_result["score"], 3),
        "top_chunk": top_result["chunk_id"],
    })
pd.DataFrame(threshold_rows).sort_values("top_score", ascending=False)

## Exercise E1: Compare free local embeddings with OpenAI's small embedder

Rebuild the sentence-window index using OpenAI's `text-embedding-3-small`, the least expensive current OpenAI embedding model, if it is available through the course service. Keep the chunk collection, questions, top-k value, and metric fixed.

- Implement an OpenAI embedding function.
- Cache embeddings locally so repeated analysis does not repeat paid calls.
- Compare retrieval hit@3 with `all-MiniLM-L6-v2`.
- Compare at least three individual rankings.
- Discuss quality, privacy, latency, cost, reproducibility, and operational dependencies.
- State which embedder you would choose for this collection and why.

If the instructor-managed endpoint does not provide embeddings, compare two free Sentence Transformer models instead.

In [ ]:
# EXERCISE E1: Implement cached alternative embeddings and the controlled comparison here.
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

## Exercise E2: Design a memory policy and test context budgets

Complete both parts.

**Part A — Context budget experiment**

- Run the evaluation set with at least three context budgets.
- Compare included evidence, citation behavior, abstention, and answer quality.
- Identify one case where more context helps and one where additional context is redundant or distracting.

**Part B — Memory policy**

- Propose at least eight candidate memory items for a course assistant.
- Classify each as working state, safe persistent preference, sensitive information, or unnecessary data.
- Decide whether to persist, retain temporarily, request explicit consent, or reject each item.
- Implement one additional safe preference plus a function that lists and deletes stored preferences.
- Explain retention, access, provenance, update, and deletion rules.

In [ ]:
# EXERCISE E2: Run the budget experiment and implement the memory-policy demonstration here.
context_budgets = [350, 700, 1400]

## Lab Takeaways

- Retrieval, context, memory, and state solve different problems.
- Chunking decisions shape what can be retrieved and cited.
- Embedding similarity ranks candidates but does not prove relevance or truth.
- Context assembly should enforce budgets, remove redundancy, and preserve provenance.
- Grounded generation needs explicit evidence, citations, abstention, and evaluation.
- Workflow state should be explicit; persistent memory should be minimal and governed.
- Retrieval and generation failures should be measured separately.